# 🧠 Cheng Figshare Brain Tumor Dataset: Complete Preprocessing & Conversion Pipeline
### Convert `.mat` (MATLAB v7.3 HDF5) to Lossless `.png` Images, Binary Masks & 1-Click Downloadable `.zip`

This notebook is tailored specifically for the **Cheng Figshare Brain Tumor MRI Dataset** (3,064 slices from 233 patients).
It is fully optimized to run either **locally** or on **Kaggle** (with GPU/CPU).

---

### 📌 Objectives & Key Features:
1. **Load `.mat` files** using `h5py` (MATLAB v7.3 HDF5 format).
2. **Min-Max Normalize** raw 16-bit MRI voxel intensities to standard 8-bit `uint8` (`.png`).
3. **Extract Ground-Truth Masks** (`tumorMask`) as clean binary images ($0$ and $255$).
4. **Integrate Official 5-Fold Cross-Validation** (`cvind.mat`) to strictly prevent patient data leakage.
5. **Generate a Master Metadata CSV** with Patient IDs (`PID`), tumor classes, and fold splits.
6. **Visual Inspection**: Display MRI scans side-by-side with masks and color overlays (Red = Tumor).
7. **PyTorch Dataset Loader**: Ready-to-use Dataset class for classification and segmentation models (U-Net, ResNet, etc.).
8. **1-Click Zip Packaging**: Automatically bundles the 6,000+ files into a single `.zip` archive so you can download it from Kaggle in seconds without browser timeouts or disconnects!


In [ ]:
import os
import glob
import shutil
import h5py
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

print("✅ All required libraries imported successfully!")


## ⚙️ 1. Directory & Environment Configuration
Automatically detects whether the notebook is running in **Kaggle** or in a **Local** environment, and sets paths accordingly.


In [ ]:
# Automatic environment detection
if os.path.exists('/kaggle/input'):
    print("Detected environment: Kaggle")
    # Search for cheng figshare directory in /kaggle/input
    possible_dirs = glob.glob('/kaggle/input/**/brainTumorDataPublic*', recursive=True)
    if possible_dirs:
        INPUT_DIR = os.path.dirname(possible_dirs[0])
    else:
        INPUT_DIR = '/kaggle/input/brain-tumor-figshare'
    OUTPUT_DIR = '/kaggle/working/cheng_figshare_processed'
else:
    print("Detected environment: Local")
    INPUT_DIR = 'datasets/cheng_figshare'
    OUTPUT_DIR = 'datasets/cheng_figshare_processed'

print(f"📁 Input Directory : {INPUT_DIR}")
print(f"📁 Output Directory: {OUTPUT_DIR}")


## 🔬 2. Inspect a Sample `.mat` File
Each file contains a MATLAB struct `cjdata` with:
- `PID`: Patient identifier string (essential for patient-stratified splits)
- `image`: $512 \times 512$ raw MRI voxel intensities (16-bit signed integer)
- `label`: $1$ = Meningioma, $2$ = Glioma, $3$ = Pituitary
- `tumorMask`: $512 \times 512$ binary mask ($1$ for tumor, $0$ for background)
- `tumorBorder`: Polygon coordinate vertices of the tumor outline


In [ ]:
# Find all patient slice .mat files
mat_files = sorted(glob.glob(os.path.join(INPUT_DIR, '**', '*.mat'), recursive=True))
data_files = [f for f in mat_files if not os.path.basename(f).startswith('cvind')]

print(f"Total .mat slice files found: {len(data_files)}")

# Inspect the very first slice
sample_file = data_files[0]
with h5py.File(sample_file, 'r') as f:
    cjdata = f['cjdata']
    pid = ''.join([chr(c[0]) for c in cjdata['PID']])
    label = int(cjdata['label'][0, 0])
    raw_img = np.array(cjdata['image'], dtype=np.float32)
    mask = np.array(cjdata['tumorMask'], dtype=np.uint8)

print(f"Sample File: {os.path.basename(sample_file)}")
print(f"  Patient ID  : {pid}")
print(f"  Class Label : {label} (1=Meningioma, 2=Glioma, 3=Pituitary)")
print(f"  Image Shape : {raw_img.shape}, Min: {raw_img.min()}, Max: {raw_img.max()}")
print(f"  Mask Shape  : {mask.shape}, Unique Values: {np.unique(mask)}")


## 📋 3. Load 5-Fold Cross Validation Indices (`cvind.mat`)
Cheng et al. provide official 5-fold cross-validation indices grouped by patient.
Loading this ensures your research can benchmark against published state-of-the-art papers without data leakage.


In [ ]:
cvind_path = os.path.join(INPUT_DIR, 'cvind.mat')
fold_dict = {}

if os.path.exists(cvind_path):
    with h5py.File(cvind_path, 'r') as f:
        cvind = np.array(f['cvind']).squeeze().astype(int)
        print(f"cvind shape: {cvind.shape}, Folds present: {np.unique(cvind)}")
        for idx, fold in enumerate(cvind, start=1):
            fold_dict[idx] = fold
    print(f"✅ Loaded fold assignments for all {len(fold_dict)} slices!")
else:
    print("⚠️ cvind.mat not found in INPUT_DIR. Fold column will be set to None.")


## 🖼️ 4. Convert `.mat` to Lossless `.png` & Generate Metadata
- **MRI Images**: Min-Max normalized to $[0, 255]$ and saved as lossless `.png`.
- **Masks**: Multiplied by $255$ to produce clean binary images ($0$ and $255$) and saved as `.png`.
- **Metadata CSV**: Generates `cheng_figshare_metadata.csv` tracking file paths, classes, patient IDs, tumor area, and folds.


In [ ]:
class_map = {1: 'meningioma', 2: 'glioma', 3: 'pituitary'}

# Create destination subdirectories
for category in ['images', 'masks']:
    for c in class_map.values():
        os.makedirs(os.path.join(OUTPUT_DIR, category, c), exist_ok=True)

metadata_records = []

for file_path in tqdm(data_files, desc="Converting .mat to .png"):
    filename = os.path.basename(file_path)
    slice_id = int(os.path.splitext(filename)[0])
    
    with h5py.File(file_path, 'r') as f:
        cjdata = f['cjdata']
        pid = ''.join([chr(c[0]) for c in cjdata['PID']])
        label = int(cjdata['label'][0, 0])
        class_name = class_map[label]
        
        # 1. Normalize image
        img_arr = np.array(cjdata['image'], dtype=np.float32)
        min_v, max_v = img_arr.min(), img_arr.max()
        norm_img = ((img_arr - min_v) / (max_v - min_v + 1e-8) * 255.0).astype(np.uint8)
        
        # 2. Extract binary mask
        mask_arr = np.array(cjdata['tumorMask'], dtype=np.uint8) * 255
        tumor_pixel_count = int(np.sum(mask_arr > 0))
        
        # Relative & absolute paths
        img_filename = f"{slice_id}.png"
        mask_filename = f"{slice_id}.png"
        
        rel_img = os.path.join('images', class_name, img_filename)
        rel_mask = os.path.join('masks', class_name, mask_filename)
        
        # Save as PNG
        Image.fromarray(norm_img).save(os.path.join(OUTPUT_DIR, rel_img))
        Image.fromarray(mask_arr).save(os.path.join(OUTPUT_DIR, rel_mask))
        
        metadata_records.append({
            'slice_id': slice_id,
            'patient_id': pid,
            'class_id': label,
            'class_name': class_name,
            'image_path': rel_img,
            'mask_path': rel_mask,
            'fold': fold_dict.get(slice_id, None),
            'tumor_area_pixels': tumor_pixel_count,
            'width': norm_img.shape[1],
            'height': norm_img.shape[0]
        })

df_meta = pd.DataFrame(metadata_records)
csv_save_path = os.path.join(OUTPUT_DIR, 'cheng_figshare_metadata.csv')
df_meta.to_csv(csv_save_path, index=False)

print(f"🎉 Conversion Completed! Processed {len(df_meta)} slices.")
print(f"📄 Metadata saved to: {csv_save_path}")


## 📊 5. Summary Statistics & Cross-Validation Verification

In [ ]:
print("=== Class Distribution ===")
print(df_meta['class_name'].value_counts())

print(f"\nTotal Unique Patients: {df_meta['patient_id'].nunique()}")

if 'fold' in df_meta.columns and df_meta['fold'].notna().all():
    print("\n=== Slices per Cross-Validation Fold ===")
    print(pd.crosstab(df_meta['fold'], df_meta['class_name'], margins=True))

df_meta.head(10)


## 🎨 6. Visualizing MRI Scans with Overlaid Tumor Masks
Displays sample scans from each class:
- **Left:** Original MRI scan
- **Middle:** Ground-truth binary segmentation mask
- **Right:** Red highlight overlay of the tumor onto the MRI scan


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for i, (cls_name, group) in enumerate(df_meta.groupby('class_name')):
    sample_row = group.iloc[0]
    
    img_p = os.path.join(OUTPUT_DIR, sample_row['image_path'])
    mask_p = os.path.join(OUTPUT_DIR, sample_row['mask_path'])
    
    mri = np.array(Image.open(img_p))
    msk = np.array(Image.open(mask_p))
    
    # Red overlay
    overlay = np.stack([mri, mri, mri], axis=-1)
    overlay[msk > 0] = [255, 50, 50]
    
    # Col 1: MRI
    axes[i, 0].imshow(mri, cmap='gray')
    axes[i, 0].set_title(f"{cls_name.capitalize()} (MRI)\nPatient ID: {sample_row['patient_id']}", fontsize=12)
    axes[i, 0].axis('off')
    
    # Col 2: Mask
    axes[i, 1].imshow(msk, cmap='gray')
    axes[i, 1].set_title(f"Tumor Mask ({sample_row['tumor_area_pixels']} px)", fontsize=12)
    axes[i, 1].axis('off')
    
    # Col 3: Overlay
    axes[i, 2].imshow(overlay)
    axes[i, 2].set_title(f"Overlay (Tumor in Red)", fontsize=12)
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()


## 🚀 7. Ready-to-Use PyTorch Dataset Class
Use this class directly in your training scripts for classification or segmentation models:


In [ ]:
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T

class ChengBrainTumorDataset(Dataset):
    """
    PyTorch Dataset for Cheng Figshare Brain Tumor MRI.
    Supports both Classification (label) and Segmentation (mask).
    """
    def __init__(self, metadata_df, base_dir, transform=None, mask_transform=None):
        self.df = metadata_df.reset_index(drop=True)
        self.base_dir = base_dir
        self.transform = transform
        self.mask_transform = mask_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img_path = os.path.join(self.base_dir, row['image_path'])
        mask_path = os.path.join(self.base_dir, row['mask_path'])
        
        image = Image.open(img_path).convert('L')
        mask = Image.open(mask_path).convert('L')
        
        if self.transform:
            image = self.transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)
            mask = (mask > 0.5).float()
            
        label = torch.tensor(row['class_id'] - 1, dtype=torch.long)
        
        return {
            'image': image,
            'mask': mask,
            'label': label,
            'patient_id': row['patient_id'],
            'slice_id': row['slice_id']
        }

print("✅ ChengBrainTumorDataset class defined and ready for training!")


## 📦 8. Bundle into a Single `.zip` Archive for Fast 1-Click Download
### ⚡ Why this step is essential:
In Kaggle, downloading an output folder that contains **6,000+ individual loose images** causes browser HTTP timeouts and drops connections repeatedly.
By packaging the entire processed directory into **one single `.zip` file**, Kaggle can serve the download at maximum network speed without any timeouts or interruptions!


In [ ]:
# Determine the zip archive target path
if os.path.exists('/kaggle/working'):
    zip_base = '/kaggle/working/cheng_figshare_processed'
else:
    zip_base = os.path.join(os.path.dirname(OUTPUT_DIR), 'cheng_figshare_processed')

print(f"📦 Compressing '{OUTPUT_DIR}' into a single .zip archive...")
print("Please wait ~15-20 seconds...")

archive_path = shutil.make_archive(
    base_name=zip_base,
    format='zip',
    root_dir=OUTPUT_DIR
)

file_size_mb = os.path.getsize(archive_path) / (1024 * 1024)
print(f"\n🎉 SUCCESS! Zip archive created: {archive_path}")
print(f"📊 Total Zip Size: {file_size_mb:.2f} MB")
print("\n👉 HOW TO DOWNLOAD:")
print("1. Look at the Kaggle right sidebar under 'Output' (or on the notebook viewer page).")
print("2. You will see 'cheng_figshare_processed.zip'.")
print("3. Click the 3 vertical dots next to it and select 'Download'.")
print("4. The entire dataset will download as one file smoothly without disconnecting!")
